# 10 — FINAL Bridge Type Engineering Decision Report

## Purpose

Consolidate the validated evidence produced by Notebooks 07, 08 and 09 into one auditable engineering decision-support package.

This stage keeps classification, condition, validation, explainability and robustness evidence separate. It does not calculate an artificial overall score, retrain or modify the frozen production model, or perform FEM/structural design.


In [1]:
from pathlib import Path
import os, json
import pandas as pd

def find_project_root():
    env_root=os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root=Path(env_root).expanduser().resolve()
        if (root/"Dataset_PlanA-B").exists(): return root
        raise FileNotFoundError(f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}")
    current=Path.cwd().resolve()
    for candidate in [current,*current.parents]:
        if (candidate/"Dataset_PlanA-B").exists(): return candidate
    raise FileNotFoundError("Project root not found. Set BRIDGE_PROJECT_ROOT.")

PROJECT_ROOT=find_project_root()
DATASET_ROOT=PROJECT_ROOT/"Dataset_PlanA-B"
OUTPUT_ROOT=PROJECT_ROOT/"Output_PlanA-B"
NB07_DIR=OUTPUT_ROOT/"07_ML_Condition_Model_Validation"
NB08_DIR=OUTPUT_ROOT/"08_Bridge_Type_Decision_Engine_Independent_Validation"
NB09_DIR=OUTPUT_ROOT/"09_ML_Bridge_Type_Selection_Explainability_Robustness"
OUTPUT_DIR=OUTPUT_ROOT/"10_FINAL_Bridge_Type_Engineering_Decision_Report"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
REPORT_FILE=OUTPUT_DIR/"10_FINAL_Bridge_Type_Engineering_Decision_Report.xlsx"
INVENTORY_FILE=OUTPUT_DIR/"10_report_data_inventory.csv"
MANIFEST_JSON=OUTPUT_DIR/"10_data_manifest.json"
MANIFEST_TXT=OUTPUT_DIR/"10_data_manifest.txt"

print("Project root:",PROJECT_ROOT)
print("Output dir:",OUTPUT_DIR)
print("Upstream 07:",NB07_DIR)
print("Upstream 08:",NB08_DIR)
print("Upstream 09:",NB09_DIR)


Project root: C:\Datenanalyse\final Project
Output dir: C:\Datenanalyse\final Project\Output_PlanA-B\10_FINAL_Bridge_Type_Engineering_Decision_Report
Upstream 07: C:\Datenanalyse\final Project\Output_PlanA-B\07_ML_Condition_Model_Validation
Upstream 08: C:\Datenanalyse\final Project\Output_PlanA-B\08_Bridge_Type_Decision_Engine_Independent_Validation
Upstream 09: C:\Datenanalyse\final Project\Output_PlanA-B\09_ML_Bridge_Type_Selection_Explainability_Robustness


## 00A — DATA SOURCE / INPUT–OUTPUT MANIFEST

| Evidence layer | Source | Transfer | Used for | Output |
|---|---|---|---|---|
| Condition validation | Notebook 07 output package | Local CSV/JSON | Condition-model evidence | Final report |
| Independent validation | Notebook 08 output package | Local CSV/JSON | Decision-engine validation | Final report |
| Explainability & robustness | Notebook 09 output package | Local CSV/JSON | Importance/sensitivity/stability evidence | Final report |
| Consolidated report | Notebook 10 dataframes | Local Excel/CSV/JSON write | Engineering evidence package | `10_FINAL_Bridge_Type_Engineering_Decision_Report.xlsx` |
| Report inventory | Actual loaded files | Generated | Provenance/audit | `10_report_data_inventory.csv` |
| Manifest | Notebook 10 configuration | Generated | Source/transfer documentation | `10_data_manifest.json/.txt` |

### Transfer chain
```text
07 Condition Validation ─────────┐
08 Independent Validation ──────┼→ 10 FINAL Engineering Decision Report
09 Explainability & Robustness ─┘
                                  ↓
Output_PlanA-B/10_FINAL_Bridge_Type_Engineering_Decision_Report
```


## 01 — Verify required upstream evidence artifacts


In [2]:
REQUIRED_INPUTS={
 "07":{"dir":NB07_DIR,"files":["07_baseline_vs_bauwerksart_model.csv","07_per_type_validation.csv","07_condition_distribution_by_bridge_type.csv","07_candidate_type_comparison.csv","07_analysis_data_inventory.csv","07_data_manifest.json","07_data_manifest.txt"]},
 "08":{"dir":NB08_DIR,"files":["08_classifier_validation_metrics.csv","08_classifier_validation_by_type.csv","08_condition_validation_metrics.csv","08_condition_validation_by_type.csv","08_counterfactual_selection_simulation.csv","08_validation_summary.csv","08_analysis_data_inventory.csv","08_data_manifest.json","08_data_manifest.txt"]},
 "09":{"dir":NB09_DIR,"files":["09_classifier_permutation_importance.csv","09_condition_permutation_importance.csv","09_dtv_sensitivity.csv","09_material_sensitivity.csv","09_location_sensitivity.csv","09_model_stability.csv","09_analysis_data_inventory.csv","09_data_manifest.json","09_data_manifest.txt"]}
}
rows=[]; missing=[]
for stage,spec in REQUIRED_INPUTS.items():
    for name in spec["files"]:
        p=spec["dir"]/name; ok=p.exists()
        rows.append({"stage":stage,"artifact":name,"path":str(p),"exists":ok,"size_bytes":p.stat().st_size if ok else None})
        if not ok: missing.append(str(p))
artifact_status=pd.DataFrame(rows)
display(artifact_status)
if missing: raise FileNotFoundError("Required upstream artifacts are missing:\n"+"\n".join(missing))
print("Upstream artifact completeness: PASS")


,stage,artifact,path,exists,size_bytes
0,07,07_baseline_vs_bauwerksart_model.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,249
1,07,07_per_type_validation.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,3006
2,07,07_condition_distribution_by_bridge_type.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,4666
3,07,07_candidate_type_comparison.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,1857
4,07,07_analysis_data_inventory.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,271
5,07,07_data_manifest.json,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,857
6,07,07_data_manifest.txt,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,971
7,08,08_classifier_validation_metrics.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,133
8,08,08_classifier_validation_by_type.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,1692
9,08,08_condition_validation_metrics.csv,C:\Datenanalyse\final Project\Output_PlanA-B\0...,True,88


Upstream artifact completeness: PASS


## 02 — Load upstream evidence tables


In [3]:
def load_csv(stage,name): return pd.read_csv(REQUIRED_INPUTS[stage]["dir"]/name)
evidence_tables={
 "07_baseline_vs_bauwerksart":load_csv("07","07_baseline_vs_bauwerksart_model.csv"),
 "07_per_type":load_csv("07","07_per_type_validation.csv"),
 "07_condition_distribution":load_csv("07","07_condition_distribution_by_bridge_type.csv"),
 "07_candidate_comparison":load_csv("07","07_candidate_type_comparison.csv"),
 "08_classifier_metrics":load_csv("08","08_classifier_validation_metrics.csv"),
 "08_classifier_by_type":load_csv("08","08_classifier_validation_by_type.csv"),
 "08_condition_metrics":load_csv("08","08_condition_validation_metrics.csv"),
 "08_condition_by_type":load_csv("08","08_condition_validation_by_type.csv"),
 "08_counterfactual":load_csv("08","08_counterfactual_selection_simulation.csv"),
 "08_validation_summary":load_csv("08","08_validation_summary.csv"),
 "09_classifier_importance":load_csv("09","09_classifier_permutation_importance.csv"),
 "09_condition_importance":load_csv("09","09_condition_permutation_importance.csv"),
 "09_dtv_sensitivity":load_csv("09","09_dtv_sensitivity.csv"),
 "09_material_sensitivity":load_csv("09","09_material_sensitivity.csv"),
 "09_location_sensitivity":load_csv("09","09_location_sensitivity.csv"),
 "09_model_stability":load_csv("09","09_model_stability.csv"),
}
for name,table in evidence_tables.items(): print(f"{name}: {table.shape}")


07_baseline_vs_bauwerksart: (2, 5)
07_per_type: (33, 5)
07_condition_distribution: (43, 7)
07_candidate_comparison: (20, 6)
08_classifier_metrics: (1, 5)
08_classifier_by_type: (33, 3)
08_condition_metrics: (1, 4)
08_condition_by_type: (33, 5)
08_counterfactual: (100, 4)
08_validation_summary: (1, 7)
09_classifier_importance: (4, 3)
09_condition_importance: (5, 3)
09_dtv_sensitivity: (5, 3)
09_material_sensitivity: (7, 3)
09_location_sensitivity: (10, 3)
09_model_stability: (3, 2)


## 03 — Evidence-layer summary and engineering boundary


In [4]:
evidence_layer_summary=pd.DataFrame([
 {"stage":"07","evidence_layer":"Condition model validation","source_directory":str(NB07_DIR),"purpose":"Condition estimation and bridge-type effect validation."},
 {"stage":"08","evidence_layer":"Independent decision-engine validation","source_directory":str(NB08_DIR),"purpose":"Independent validation of classification and condition components."},
 {"stage":"09","evidence_layer":"Explainability and robustness","source_directory":str(NB09_DIR),"purpose":"Feature importance, sensitivity and model stability diagnostics."},
])
display(evidence_layer_summary)
architecture_statement={"inputs":["latitude","longitude","dtv","bauwerkstoff"],"type_target":"bauwerksart","condition_target":"zustandsnote","artificial_overall_score":False,"production_model_modified":False,"fem_or_structural_design_performed":False,"counterfactual_alternative_type_ground_truth":False}
display(pd.DataFrame([architecture_statement]))


,stage,evidence_layer,source_directory,purpose
0,07,Condition model validation,C:\Datenanalyse\final Project\Output_PlanA-B\0...,Condition estimation and bridge-type effect va...
1,08,Independent decision-engine validation,C:\Datenanalyse\final Project\Output_PlanA-B\0...,Independent validation of classification and c...
2,09,Explainability and robustness,C:\Datenanalyse\final Project\Output_PlanA-B\0...,"Feature importance, sensitivity and model stab..."


,inputs,type_target,condition_target,artificial_overall_score,production_model_modified,fem_or_structural_design_performed,counterfactual_alternative_type_ground_truth
0,"[latitude, longitude, dtv, bauwerkstoff]",bauwerksart,zustandsnote,False,False,False,False


## 04 — Export final evidence package


In [5]:
with pd.ExcelWriter(REPORT_FILE,engine="openpyxl") as writer:
    evidence_layer_summary.to_excel(writer,sheet_name="00_Evidence_Summary",index=False)
    for name,table in evidence_tables.items(): table.to_excel(writer,sheet_name=name[:31],index=False)

inventory_rows=[]
for stage,spec in REQUIRED_INPUTS.items():
    for name in spec["files"]:
        p=spec["dir"]/name
        if p.suffix.lower()==".csv":
            header=pd.read_csv(p,nrows=0)
            with open(p,encoding="utf-8-sig",errors="replace") as f: row_count=max(sum(1 for _ in f)-1,0)
            inventory_rows.append({"stage":stage,"artifact":name,"rows":row_count,"columns":len(header.columns),"source_path":str(p)})
        else: inventory_rows.append({"stage":stage,"artifact":name,"rows":None,"columns":None,"source_path":str(p)})
report_inventory=pd.DataFrame(inventory_rows)
report_inventory.to_csv(INVENTORY_FILE,index=False,encoding="utf-8-sig")
manifest={"stage":10,"notebook":"10_FINAL_Bridge_Type_Engineering_Decision_Report","upstream_stages":[7,8,9],"upstream_directories":{"07":str(NB07_DIR),"08":str(NB08_DIR),"09":str(NB09_DIR)},"report_file":str(REPORT_FILE),"inventory_file":str(INVENTORY_FILE),"artificial_overall_score":False,"production_model_modified":False,"fem_or_structural_design_performed":False}
MANIFEST_JSON.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding="utf-8")
MANIFEST_TXT.write_text("Notebook 10 — FINAL Bridge Type Engineering Decision Report\n===========================================================\n\n"+f"PROJECT_ROOT: {PROJECT_ROOT}\nOUTPUT_DIR: {OUTPUT_DIR}\n\nUPSTREAM 07: {NB07_DIR}\nUPSTREAM 08: {NB08_DIR}\nUPSTREAM 09: {NB09_DIR}\n\nThe report keeps evidence layers separate; no artificial overall score, retraining or FEM/design is performed.\n\nREPORT: {REPORT_FILE}\nINVENTORY: {INVENTORY_FILE}\n",encoding="utf-8")
print("10 STATUS: COMPLETE")
print("Output directory:",OUTPUT_DIR)
print("Report:",REPORT_FILE)
print("Data inventory:",INVENTORY_FILE)
print("Data manifest:",MANIFEST_TXT)


10 STATUS: COMPLETE
Output directory: C:\Datenanalyse\final Project\Output_PlanA-B\10_FINAL_Bridge_Type_Engineering_Decision_Report
Report: C:\Datenanalyse\final Project\Output_PlanA-B\10_FINAL_Bridge_Type_Engineering_Decision_Report\10_FINAL_Bridge_Type_Engineering_Decision_Report.xlsx
Data inventory: C:\Datenanalyse\final Project\Output_PlanA-B\10_FINAL_Bridge_Type_Engineering_Decision_Report\10_report_data_inventory.csv
Data manifest: C:\Datenanalyse\final Project\Output_PlanA-B\10_FINAL_Bridge_Type_Engineering_Decision_Report\10_data_manifest.txt
